# 🌐 KHÁM PHÁ & PHÂN RÃ TOÀN BỘ URL CORPUS THEO DẤU GẠCH CHÉO ("/")
### ViBioMIR / Road to AI 2026 — Comprehensive URL Path & Hierarchy Explorer

> **Mục đích Notebook:**
> 1. **Tải & Khảo sát toàn bộ 1.22M URLs** y khoa từ Ban Tổ chức cuộc thi (`links_corpus_vi_en_zh.parquet`).
> 2. **Phân rã đa cấp độ theo dấu `/`**: Tách nhỏ từng URL thành `domain`, `depth` (độ sâu thư mục), `seg_1` (danh mục lớn), `seg_2` (tiểu mục/chuyên khoa), `seg_3`, `leaf` (slug/định danh bài viết) và `extension` (`.htm`, `.html`, `.php`).
> 3. **Phân tích Cây Phân Cấp (URL Hierarchy Tree)**: Tự động gom nhóm cây thư mục theo từng Domain nguồn (`vinmec.com`, `suckhoedoisong.vn`, `120ask.com`, `bachmai.gov.vn`...).
> 4. **Đối chiếu Trạng Thái Cào (Crawl Coverage)**: Đánh dấu URL nào đã cào thành công (`✅ Đã cào`) vs URL nào còn thiếu (`⏳ Chưa cào`).
> 5. **Phát hiện URL Rác / Bất thường**: Nhận diện trang chủ, trang phân trang (`/page/2`), trang thẻ tag rỗng để tối ưu hóa kho dữ liệu.
> 6. **Giao diện Trực quan & Tương tác**: Tra cứu theo từ khóa, lọc theo domain, xem thẻ bài viết có link bấm trực tiếp ↗.


In [1]:
# 1. Khởi tạo môi trường & Cấu hình Giao diện Tự Động Xuống Dòng (Text Wrap)
import os
import sys
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import HTML, display

# Cấu hình đường dẫn dự án
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "src").exists() else CURRENT_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# CSS tối ưu hiển thị: Không bị tràn thanh cuộn ngang, giao diện hiện đại sáng rõ
display(HTML("""
<style>
    .output_text pre, .output_subarea pre, div.output_area pre {
        white-space: pre-wrap !important;
        word-wrap: break-word !important;
        word-break: break-word !important;
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif !important;
    }
    .dataframe td, .dataframe th {
        max-width: 350px;
        word-wrap: break-word;
        white-space: pre-wrap !important;
    }
</style>
"""))

print(f"✅ Đã sẵn sàng! Thư mục làm việc: {PROJECT_ROOT}")


✅ Đã sẵn sàng! Thư mục làm việc: /home/thienhb/Workspace/med-doc-retrieval


---
## 📥 1. Nạp Toàn Bộ Danh Sách URLs & Đối Chiếu Trạng Thái Cào

Tải file `links_corpus_vi_en_zh.parquet` (1.220.598 URLs) từ BTC và nạp danh sách các ID đã cào thành công để đối chiếu độ phủ.

In [2]:
# Tải danh sách URLs và danh sách đã cào
links_candidates = [
    PROJECT_ROOT / "data/raw/vibio_mir/links_corpus_vi_en_zh.parquet",
    PROJECT_ROOT / "data/raw/vibio_mir/links_corpus.parquet",
    PROJECT_ROOT / "links_corpus_vi_en_zh.parquet",
]
links_file = next((p for p in links_candidates if p.exists()), None)

if not links_file:
    raise FileNotFoundError("Không tìm thấy file links_corpus. Hãy đảm bảo file nằm trong data/raw/vibio_mir/")

links_table = pq.read_table(str(links_file))
urls_df = links_table.to_pandas()

# Nạp danh sách đã cào (scraped IDs)
scraped_file = PROJECT_ROOT / "data/processed/scraped_ids.parquet"
scraped_set = set()
if scraped_file.exists():
    scraped_table = pq.read_table(str(scraped_file))
    scraped_set = set(scraped_table["doc_id"].to_pylist())

total_urls = len(urls_df)
crawled_count = sum(1 for doc_id in urls_df["id"] if doc_id in scraped_set)
coverage_pct = (crawled_count / total_urls) * 100 if total_urls > 0 else 0

print("=" * 65)
print(f"📊 TỔNG QUAN TẬP DỮ LIỆU URL:")
print(f"  • Tổng số URLs BTC cung cấp:  {total_urls:,} links")
print(f"  • Số lượng bài đã cào về:     {crawled_count:,} articles ({coverage_pct:.2f}%)")
print(f"  • Số lượng bài chưa cào/thiếu: {total_urls - crawled_count:,} articles")
print("=" * 65)


📊 TỔNG QUAN TẬP DỮ LIỆU URL:
  • Tổng số URLs BTC cung cấp:  1,220,598 links
  • Số lượng bài đã cào về:     807,582 articles (66.16%)
  • Số lượng bài chưa cào/thiếu: 413,016 articles


---
## 🔪 2. Phân Rã Toàn Bộ URLs Theo Dấu Gạch Chéo ("/")

Hàm `dissect_urls_df` sử dụng kỹ thuật xử lý phân tách chuỗi tối ưu, phân rã 1.22M URLs thành các trường:
- `domain`: Tên miền chính (Host)
- `depth`: Độ sâu thư mục (tổng số lượng phân đoạn ngăn cách bởi `/`)
- `seg_1`: Phân đoạn cấp 1 (Thư mục gốc / Ngôn ngữ / Chuyên mục lớn nhất)
- `seg_2`: Phân đoạn cấp 2 (Tiểu mục / Chuyên khoa lâm sàng con)
- `seg_3`: Phân đoạn cấp 3 (Chi tiết hơn)
- `leaf`: Tên file hoặc Slug bài viết cuối cùng
- `extension`: Đuôi mở rộng (`.htm`, `.html`, `.php`, `none`)
- `is_crawled`: Đã cào về hay chưa (`True` / `False`)


In [11]:
import time
from src.crawler.url_parser import dissect_urls_df

print(f"🚀 Bắt đầu phân rã {len(urls_df):,} URLs theo từng cấp dấu gạch chéo (/)...\n")
t0 = time.time()

# Phân rã toàn bộ URLs
dissected_df = dissect_urls_df(
    urls_or_df=urls_df,
    id_col="id",
    url_col="url",
    scraped_ids=scraped_set,
)
elapsed = time.time() - t0

print(f"🎉 Hoàn tất phân rã {len(dissected_df):,} URLs trong {elapsed:.2f} giây! ({len(dissected_df)/elapsed:,.0f} URLs/giây)")

# Hiển thị 10 dòng mẫu sau khi bóc tách
cols_to_show = ["id", "domain", "depth", "seg_1", "seg_2", "leaf", "extension", "is_crawled"]
display(dissected_df[cols_to_show].head(50))


🚀 Bắt đầu phân rã 1,220,598 URLs theo từng cấp dấu gạch chéo (/)...

🎉 Hoàn tất phân rã 1,220,598 URLs trong 11.08 giây! (110,204 URLs/giây)


,id,domain,depth,seg_1,seg_2,leaf,extension,is_crawled
0,583,special.vietnamplus.vn,1,benh_vien_nhan_ai,,benh_vien_nhan_ai,,True
1,584,bachmai.gov.vn,2,bai-viet,-nguoi-dan-se-duoc-thu-huong-chat-luong-kham-c...,-nguoi-dan-se-duoc-thu-huong-chat-luong-kham-c...,,True
2,585,bachmai.gov.vn,2,bai-viet,-roi-loan-su-thich-ung-va-nhung-he-luy--9826,-roi-loan-su-thich-ung-va-nhung-he-luy--9826,,True
3,586,bachmai.gov.vn,2,bai-viet,1-4-nguoi-truong-thanh-bi-tang-huyet-ap,1-4-nguoi-truong-thanh-bi-tang-huyet-ap,,True
4,587,bachmai.gov.vn,2,bai-viet,1-5-phai-manh-phan-nan-ve-tinh-trang-tren-bao-...,1-5-phai-manh-phan-nan-ve-tinh-trang-tren-bao-...,,True
5,588,bachmai.gov.vn,2,bai-viet,1-5-ty-dong-tri-lieu-ung-thu-bang-vac-xin-giam...,1-5-ty-dong-tri-lieu-ung-thu-bang-vac-xin-giam...,,True
6,589,bachmai.gov.vn,2,bai-viet,1-500-nhan-vien-y-te-benh-vien-bach-mai-duoc-t...,1-500-nhan-vien-y-te-benh-vien-bach-mai-duoc-t...,,True
7,590,bachmai.gov.vn,2,bai-viet,1-dieu-thuoc-la-co-7000-chat-doc-va-khoang-70-...,1-dieu-thuoc-la-co-7000-chat-doc-va-khoang-70-...,,True
8,591,bachmai.gov.vn,2,bai-viet,1-nguoi-tu-vong-2-vo-chong-nguy-kich-vi-uong-b...,1-nguoi-tu-vong-2-vo-chong-nguy-kich-vi-uong-b...,,True
9,592,bachmai.gov.vn,2,bai-viet,10-cach-ngan-ngua-ung-thu,10-cach-ngan-ngua-ung-thu,,True


---
## 📏 3. Phân Bố Độ Sâu Đường Dẫn (URL Path Depth Distribution)

Phân bố độ sâu (số lượng dấu gạch chéo `/`) cho biết cấu trúc URL của toàn bộ kho tài liệu:
- **Depth 0**: Trang chủ hoặc thư mục gốc (`/`)
- **Depth 1**: Đường dẫn phẳng dạng `/{slug}.html` hoặc `/{pmid}` (ví dụ: Thanh Niên, Sức Khỏe & Đời Sống, PubMed)
- **Depth 2**: Đường dẫn phân cấp 1 tầng dạng `/{category}/{slug}` (ví dụ: `120ask.com/question/...`, `bachmai.gov.vn/bai-viet/...`)
- **Depth 3**: Đường dẫn phân cấp 2 tầng dạng `/{lang}/{category}/{slug}` (ví dụ: Vinmec `vie/bai-viet/...`)
- **Depth 4+**: Đường dẫn phân cấp sâu (chuyên khoa nhiều tầng hoặc CMS lồng ghép)


In [4]:
depth_summary = (
    dissected_df.groupby("depth")
    .agg(
        so_luong_url=("id", "count"),
        so_luong_da_cao=("is_crawled", lambda x: int(x.sum())),
        domain_dai_dien=("domain", "nunique"),
        mau_url=("url", "first"),
    )
    .reset_index()
)

depth_summary["ty_le_url_%"] = (depth_summary["so_luong_url"] / len(dissected_df) * 100).round(2)
depth_summary["ty_le_da_cao_%"] = (
    depth_summary["so_luong_da_cao"] / depth_summary["so_luong_url"] * 100
).round(1)

print("📊 BẢNG PHÂN BỐ ĐỘ SÂU URL (PATH DEPTH):")
display(depth_summary[["depth", "so_luong_url", "ty_le_url_%", "so_luong_da_cao", "ty_le_da_cao_%", "domain_dai_dien", "mau_url"]])


📊 BẢNG PHÂN BỐ ĐỘ SÂU URL (PATH DEPTH):


,depth,so_luong_url,ty_le_url_%,so_luong_da_cao,ty_le_da_cao_%,domain_dai_dien,mau_url
0,0,3,0.00,3,100.0,3,https://medlatec.vn/
1,1,740706,60.68,542768,73.3,40,http://special.vietnamplus.vn/benh_vien_nhan_ai
2,2,324565,26.59,141438,43.6,30,https://bachmai.gov.vn/bai-viet/-nguoi-dan-se-...
3,3,116606,9.55,84819,72.7,45,https://baoquangtri.vn/Multimedia/202202/nhung...
4,4,36300,2.97,36146,99.6,10,https://baoquangtri.vn/Multimedia/emagazine/20...
5,5,673,0.06,670,99.6,4,http://www.cnkang.com/dzjk/mrbj/ysmr/200708/99...
6,6,1703,0.14,1699,99.8,2,http://www.cnkang.com/dzjk/mrbj/xfyz/mfhf/2007...
7,7,40,0.00,38,95.0,1,http://www.cnkang.com/dzjk/mrbj/cztd/llcz/mj/2...
8,8,2,0.00,1,50.0,1,http://www.cnkang.com/dzjk/shjk/shzx/ys/swyy/y...


---
## 🏛️ 4. Thống Kê Theo Từng Domain & Độ Sâu Trung Bình

Xem danh sách 25 domain có số lượng bài lớn nhất, độ sâu đường dẫn trung bình, phân đoạn cấp 1 (`seg_1`) phổ biến nhất và tỷ lệ cào thành công.

In [5]:
domain_summary = (
    dissected_df.groupby("domain")
    .agg(
        tong_urls=("id", "count"),
        da_cao=("is_crawled", lambda x: int(x.sum())),
        do_sau_tb=("depth", lambda x: round(x.mean(), 1)),
        do_sau_max=("depth", "max"),
        seg1_pho_bien=("seg_1", lambda x: x.mode()[0] if not x.empty and x.mode()[0] else "(root)"),
        vi_du_url=("url", "first"),
    )
    .reset_index()
    .sort_values(by="tong_urls", ascending=False)
)

domain_summary["ty_le_cao_%"] = (domain_summary["da_cao"] / domain_summary["tong_urls"] * 100).round(1)

print(f"Tổng số domain ghi nhận: {len(domain_summary):,} domains\n")
display(domain_summary[["domain", "tong_urls", "da_cao", "ty_le_cao_%", "do_sau_tb", "do_sau_max", "seg1_pho_bien", "vi_du_url"]].head(25))


Tổng số domain ghi nhận: 94 domains



,domain,tong_urls,da_cao,ty_le_cao_%,do_sau_tb,do_sau_max,seg1_pho_bien,vi_du_url
53,pubmed.ncbi.nlm.nih.gov,240000,55734,23.2,1.0,1,1001981,https://pubmed.ncbi.nlm.nih.gov/11197/
59,suckhoecongdongonline.vn,154503,152091,98.4,1.0,1,(root),https://suckhoecongdongonline.vn/
76,www.120ask.com,98160,66880,68.1,2.0,2,question,http://www.120ask.com/question/1000036.htm
60,suckhoedoisong.vn,85823,82859,96.5,1.0,1,-tai-chong-vung-ve-1692341.htm,https://suckhoedoisong.vn/-tai-chong-vung-ve-1...
47,nhathuoclongchau.com.vn,79598,13,0.0,2.0,2,bai-viet,https://nhathuoclongchau.com.vn/bai-viet/1-ban...
64,thanhnien.vn,72111,70831,98.2,1.0,1,-phu-huynh-anh-bo-quen-doi-mat-cua-con-nho-185...,https://thanhnien.vn/-phu-huynh-anh-bo-quen-do...
79,www.cnkang.com,40046,39772,99.3,3.2,8,dise,http://www.cnkang.com/cm/201404/1145800.html
1,ask.39.net,32772,26335,80.4,2.0,2,question,https://ask.39.net/question/100015049.html
42,laodong.vn,31282,10,0.0,2.0,2,suc-khoe,https://laodong.vn/an/nhung-mon-an-khong-nen-t...
50,phunusuckhoe.giadinhonline.vn,27577,27472,99.6,1.0,1,1-6-benh-nhan-mac-ung-thu-deu-do-viem-5-vi-tri...,https://phunusuckhoe.giadinhonline.vn/1-6-benh...


---
## 🌳 5. Cây Thư Mục Phân Cấp (URL Hierarchy Tree) Của Từng Domain

Nhập bất kỳ domain nào (ví dụ: `www.vinmec.com`, `suckhoedoisong.vn`, `www.120ask.com`, `bachmai.gov.vn`, `thanhnien.vn`...)
để xem cây thư mục phân nhánh theo các dấu gạch chéo `/` kèm số lượng và tỷ lệ cào.

In [7]:
from src.crawler.url_parser import get_url_hierarchy_tree

def display_domain_tree(target_domain: str, top_n_seg1: int = 10, top_n_seg2: int = 4):
    tree = get_url_hierarchy_tree(dissected_df, domain=target_domain, top_n_seg1=top_n_seg1, top_n_seg2=top_n_seg2)
    if tree["total_docs"] == 0:
        print(f"❌ Không tìm thấy dữ liệu cho domain: '{target_domain}'")
        return

    html_nodes = ""
    for cat in tree["categories"]:
        seg1_name = cat["segment"]
        cnt = cat["count"]
        ratio = cat["ratio"]
        crawled = cat["crawled"]
        crawled_badge = f"<span style='background:#10b981; color:white; font-size:11px; padding:1px 6px; border-radius:3px; margin-left:6px;'>Đã cào: {crawled:,} ({crawled/cnt*100:.1f}%)</span>" if crawled is not None else ""

        sub_html = ""
        for sub in cat["sub_categories"]:
            sub_html += f"""
            <li style='margin:4px 0; font-size:12.5px; color:#334155;'>
                📁 <b>/{seg1_name}/{sub['segment']}/</b> : <b style='color:#0f172a;'>{sub['count']:,}</b> bài ({sub['ratio']}% của nhánh)
            </li>
            """

        html_nodes += f"""
        <div style='margin-bottom:12px; background:#f8fafc; border:1px solid #e2e8f0; border-left:4px solid #2563eb; border-radius:6px; padding:10px 14px;'>
            <div style='font-size:14px; font-weight:bold; color:#1e40af;'>
                📂 /{seg1_name}/ &nbsp;—&nbsp; <span style='color:#0f172a;'>{cnt:,} bài ({ratio}% domain)</span> {crawled_badge}
            </div>
            {f"<ul style='margin:6px 0 0 16px; padding-left:10px;'>{sub_html}</ul>" if sub_html else "<div style='font-size:12px; color:#64748b; margin-top:4px; font-style:italic;'>Không có thư mục con (các bài viết nằm trực tiếp dưới nhánh này).</div>"}
        </div>
        """

    tree_card = f"""
    <div style='border:1px solid #cbd5e1; border-radius:8px; padding:16px; margin:14px 0; background:#ffffff !important; color:#0f172a !important; font-family:-apple-system, BlinkMacSystemFont, \"Segoe UI\", Roboto, sans-serif;'>
        <div style='border-bottom:2px solid #e2e8f0; padding-bottom:10px; margin-bottom:14px;'>
            <h3 style='margin:0; color:#0f172a;'>🌳 Cây Phân Cấp URL (Hierarchy Tree): <span style='color:#2563eb;'>{target_domain}</span></h3>
            <div style='font-size:12px; color:#64748b; margin-top:4px;'>
                Tổng số URLs: <b>{tree['total_docs']:,}</b> | Đã cào: <b>{tree['total_crawled']:,}</b> ({tree['crawl_rate']}%) | Phân loại theo thư mục sau dấu <code>/</code>
            </div>
        </div>
        {html_nodes}
    </div>
    """
    display(HTML(tree_card))

# === ĐỔI TÊN DOMAIN BẠN MUỐN KHẢO SÁT Ở ĐÂY ===
CHOSEN_DOMAIN = "www.vinmec.com"  # Thử: "www.vinmec.com", "www.120ask.com", "bachmai.gov.vn", "suckhoedoisong.vn"
display_domain_tree(CHOSEN_DOMAIN)


---
## 🔍 6. Bộ Lọc & Tra Cứu URLs Tương Tác (Interactive URL Explorer)

Lọc nhanh URLs theo bất kỳ tiêu chí nào:
- `domain`: Lọc theo tên miền (hoặc để trống `""` để tìm toàn bộ)
- `seg_1`: Lọc theo chuyên mục cấp 1
- `keyword`: Tìm kiếm từ khóa xuất hiện trong chuỗi URL (ví dụ: `dieu-tri`, `ung-thu`, `gan`, `tim-mach`)
- `only_missing`: Chỉ hiển thị những URLs **chưa cào** để kiểm tra lý do bị sót


In [8]:
def explore_urls(
    domain: str = "",
    seg_1: str = "",
    keyword: str = "",
    min_depth: int = 1,
    max_depth: int = 10,
    only_missing: bool = False,
    limit: int = 10,
):
    sub = dissected_df
    if domain.strip():
        sub = sub[sub["domain"].str.contains(domain.strip().lower(), na=False)]
    if seg_1.strip():
        sub = sub[sub["seg_1"] == seg_1.strip()]
    if keyword.strip():
        sub = sub[sub["url"].str.contains(keyword.strip().lower(), na=False)]
    if only_missing:
        sub = sub[sub["is_crawled"] == False]
    
    sub = sub[(sub["depth"] >= min_depth) & (sub["depth"] <= max_depth)]
    total_matched = len(sub)
    
    print(f"🔎 Tìm thấy {total_matched:,} URLs thỏa mãn điều kiện lọc. Hiển thị {min(limit, total_matched)} kết quả đầu tiên:\n")
    
    cards_html = ""
    for _, row in sub.head(limit).iterrows():
        crawled_badge = "<span style='background:#10b981; color:white; font-size:11px; font-weight:bold; padding:2px 8px; border-radius:4px;'>✅ ĐÃ CÀO</span>" if row['is_crawled'] else "<span style='background:#ef4444; color:white; font-size:11px; font-weight:bold; padding:2px 8px; border-radius:4px;'>⏳ CHƯA CÀO</span>"
        
        # Tô màu phân rã từng đoạn URL
        seg_crumbs = f"<b style='color:#2563eb;'>{row['domain']}</b>"
        if row['seg_1']:
            seg_crumbs += f" &nbsp;/&nbsp; <span style='background:#e0e7ff; color:#3730a3; padding:1px 6px; border-radius:3px;'>{row['seg_1']}</span>"
        if row['seg_2']:
            seg_crumbs += f" &nbsp;/&nbsp; <span style='background:#fef3c7; color:#92400e; padding:1px 6px; border-radius:3px;'>{row['seg_2']}</span>"
        if row['leaf']:
            seg_crumbs += f" &nbsp;/&nbsp; <span style='background:#f1f5f9; color:#0f172a; padding:1px 6px; border-radius:3px;'>{row['leaf']}</span>"

        cards_html += f"""
        <div style='background:#ffffff; border:1px solid #cbd5e1; border-radius:6px; padding:12px; margin-bottom:10px; box-shadow:0 1px 3px rgba(0,0,0,0.05);'>
            <div style='display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;'>
                <div style='font-size:12px;'>
                    <span style='background:#0f172a; color:white; font-weight:bold; padding:1px 6px; border-radius:3px;'>ID: {row['id']}</span>
                    <span style='background:#e2e8f0; color:#334155; font-size:11px; padding:1px 6px; border-radius:3px; margin-left:6px;'>Độ sâu Depth: {row['depth']}</span>
                </div>
                <div>{crawled_badge}</div>
            </div>
            <div style='font-size:13px; line-height:1.6; margin:6px 0;'>
                {seg_crumbs}
            </div>
            <div style='font-size:12px; margin-top:4px;'>
                🔗 <a href='{row['url']}' target='_blank' style='color:#0d6efd; text-decoration:none;'>{row['url']} ↗</a>
            </div>
        </div>
        """
    
    container = f"""
    <div style='background:#f8fafc; border:1px solid #e2e8f0; border-radius:8px; padding:16px;'>
        {cards_html if cards_html else '<i style="color:#64748b;">Không có URL nào khớp với điều kiện tìm kiếm.</i>'}
    </div>
    """
    display(HTML(container))

# THỬ NGHIỆM TRA CỨU: (Có thể thay đổi các tham số)
explore_urls(
    domain="www.vinmec.com",     # Nhập domain muốn lọc (hoặc để trống "")
    keyword="viem-gan",          # Nhập từ khóa trong link (ví dụ: "ung-thu", "viem-gan", "tieu-duong")
    only_missing=False,          # True nếu chỉ muốn xem bài chưa cào
    limit=5,
)


🔎 Tìm thấy 197 URLs thỏa mãn điều kiện lọc. Hiển thị 5 kết quả đầu tiên:



---
## 🧹 7. Phát Hiện URL Rác & URL Bất Thường (Homepage / Phân Trang / Tag)

Tự động rà soát toàn bộ kho URLs để phát hiện:
1. **Homepage / Root URLs**: Các đường dẫn chỉ có tên miền gốc (`/`) hoặc `/index.html`.
2. **Pagination URLs**: Các link phân trang (`/page/2`, `?p=`, `?page=`) - đây không phải là bài viết y khoa mà là danh sách trang con.
3. **Missing Slugs**: URLs không có tên bài viết hoặc rỗng.


In [9]:
# 1. Rà soát Homepage URLs
homepages_df = dissected_df[dissected_df["is_homepage"] == True]

# 2. Rà soát Pagination URLs
pagination_df = dissected_df[dissected_df["is_pagination"] == True]

# 3. Rà soát URL không có phần mở rộng và độ sâu = 0
flat_root_df = dissected_df[dissected_df["depth"] == 0]

print("=" * 65)
print(f"🚨 KẾT QUẢ RÀ SOÁT CHẤT LƯỢNG URLs TOÀN BỘ CORPUS:")
print(f"  • Số URLs là Trang chủ (Homepages):    {len(homepages_df):,} URLs")
print(f"  • Số URLs là Trang phân trang (Pages): {len(pagination_df):,} URLs")
print(f"  • Số URLs Root cấp 0 (Flat root):     {len(flat_root_df):,} URLs")
print("=" * 65)

if not pagination_df.empty:
    print("\nVí dụ 5 URLs phân trang phát hiện được:")
    for u in pagination_df["url"].head(5):
        print(f"  ⚠️ {u}")

if not homepages_df.empty:
    print("\nVí dụ URLs trang chủ phát hiện được:")
    for u in homepages_df["url"].head(5):
        print(f"  🏠 {u}")


🚨 KẾT QUẢ RÀ SOÁT CHẤT LƯỢNG URLs TOÀN BỘ CORPUS:
  • Số URLs là Trang chủ (Homepages):    3 URLs
  • Số URLs là Trang phân trang (Pages): 0 URLs
  • Số URLs Root cấp 0 (Flat root):     3 URLs

Ví dụ URLs trang chủ phát hiện được:
  🏠 https://medlatec.vn/
  🏠 https://suckhoecongdongonline.vn/
  🏠 https://baidianfeng.familydoctor.com.cn/


---
## 💾 8. Xuất Bảng Phân Rã URLs Ra Parquet (Tùy Chọn)

Lưu bảng dữ liệu đã phân rã toàn bộ 1.22M URLs thành file Parquet (`data/processed/urls_dissected.parquet`, dung lượng chỉ ~32 MB) để tái sử dụng ngay lập tức cho các tác vụ lọc và phân loại văn bản tiếp theo.

In [10]:
export_path = PROJECT_ROOT / "data/processed/urls_dissected.parquet"
export_path.parent.mkdir(parents=True, exist_ok=True)

# Chọn các cột cần lưu
export_cols = ["id", "domain", "depth", "seg_1", "seg_2", "seg_3", "leaf", "extension", "is_homepage", "is_pagination", "is_crawled", "url"]
dissected_df[export_cols].to_parquet(str(export_path), index=False, compression="zstd")

size_mb = export_path.stat().st_size / (1024 * 1024)
print(f"🎉 Đã xuất thành công {len(dissected_df):,} URLs phân rã vào: {export_path}")
print(f"💾 Dung lượng file: {size_mb:.2f} MB (nén ZSTD siêu nhẹ)")


🎉 Đã xuất thành công 1,220,598 URLs phân rã vào: /home/thienhb/Workspace/med-doc-retrieval/data/processed/urls_dissected.parquet
💾 Dung lượng file: 56.50 MB (nén ZSTD siêu nhẹ)
